# M0 answer-only audit v2

Revised baseline: 1,260 non-final prompts, shared Answer: format, 128-token generation cap and per-response EOS/cap logging. Preserve the earlier 32-token run. Push the v2 code first, then connect this notebook to a Kaggle GPU kernel.
Use answer_scoring.answer_matches for v2 answer counts; exact_matches is the unchanged legacy whole-response diagnostic.

In [1]:
import os
import subprocess
import sys
import tempfile
from pathlib import Path

# This path exists only in the remote Kaggle runtime.
remote_root = Path('/kaggle/working')
assert remote_root.is_dir(), 'Connect to the Kaggle remote kernel first.'
repo_dir = Path(tempfile.mkdtemp(prefix='m0-audit-source-', dir=remote_root)) / 'repo'
subprocess.run(['git', 'clone', '--branch', 'main',
    'https://github.com/DevulapalliTharun/persistent-llm-unlearning.git', str(repo_dir)], check=True)
os.chdir(repo_dir)
print('Repository commit:', subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip())
assert Path('configs/evaluation/m0_knowledge_audit_v2.json').is_file(), 'Push the audit files first.'

Cloning into '/kaggle/working/m0-audit-source-7ynhle4g/repo'...


Repository commit: b2df70757ed57b53270913ecd453d5fc36a7ac12


In [2]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet',
    '-r', 'requirements/m0-smoke.txt'], check=True)

CompletedProcess(args=['/usr/bin/python3', '-m', 'pip', 'install', '--quiet', '-r', 'requirements/m0-smoke.txt'], returncode=0)

In [3]:
import json

# Plan-only verifies the candidate without loading any model.
plan_dir = Path(tempfile.mkdtemp(prefix='m0-audit-v2-plan-', dir=remote_root))
subprocess.run([sys.executable, '-m', 'src.m0_knowledge_audit',
    '--config', 'configs/evaluation/m0_knowledge_audit_v2.json',
    '--output-dir', str(plan_dir), '--plan-only'], check=True)
plan = json.loads((plan_dir / 'plan.json').read_text())
assert plan['planned_predictions'] == 1260
assert plan['knowledge_units'] == 180
print('Candidate checksums verified; ready for GPU inference.')

{
  "config": {
    "protocol_version": "answer-only-v2",
    "model_id": "Qwen/Qwen3-0.6B",
    "model_revision": "c1899de289a04d12100db370d81485cdf75e47ca",
    "bundle": "data/candidates/pilot-v1/seed-42",
    "expected_checksum_index_sha256": "1a70624618acb6a849f1605766a79eef0f66f8b0aab938ed500152d1705ebe05",
    "prompt_files": [
      "m1_dev.jsonl",
      "n_rel_train.jsonl"
    ],
    "batch_size": 16,
    "max_new_tokens": 128,
    "seed": 42,
    "thinking_enabled": false,
    "do_sample": false,
    "system_prompt": "Answer the question using one line: Answer: <answer>. Replace <answer> with only the requested author name, book title, prize name, year, or integer. If you do not know, reply Answer: Unknown. Do not explain, repeat the question, or list alternatives."
  },
  "config_sha256": "6c871ad523fae774de9dd9c27883ce33086955cea4c7362f710b886ec575597e",
  "input_sha256": {
    "m1_dev.jsonl": "60fe576ff6216215b97293d19d34a4f81fac3cfc6b59da1ba8f2c46a933be78c",
    "n_rel_tr

In [4]:
# A new directory preserves any previous audit run.
output_dir = Path(tempfile.mkdtemp(prefix='m0-knowledge-audit-v2-', dir=remote_root))
print('Audit outputs:', output_dir, flush=True)
subprocess.run([sys.executable, '-u', '-m', 'src.m0_knowledge_audit',
    '--config', 'configs/evaluation/m0_knowledge_audit_v2.json',
    '--output-dir', str(output_dir)], check=True)

Audit outputs: /kaggle/working/m0-knowledge-audit-v2-n1fck5ni


Loading weights: 100%|██████████| 311/311 [00:00<00:00, 427.34it/s]
[transformers] The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Audited 16/1260 prompts
Audited 32/1260 prompts
Audited 48/1260 prompts
Audited 64/1260 prompts
Audited 80/1260 prompts
Audited 96/1260 prompts
Audited 112/1260 prompts
Audited 128/1260 prompts
Audited 144/1260 prompts
Audited 160/1260 prompts
Audited 176/1260 prompts
Audited 192/1260 prompts
Audited 208/1260 prompts
Audited 224/1260 prompts
Audited 240/1260 prompts
Audited 256/1260 prompts
Audited 272/1260 prompts
Audited 288/1260 prompts
Audited 304/1260 prompts
Audited 320/1260 prompts
Audited 336/1260 prompts
Audited 352/1260 prompts
Audited 368/1260 prompts
Audited 384/1260 prompts
Audited 400/1260 prompts
Audited 416/1260 prompts
Audited 432/1260 prompts
Audited 448/1260 prompts
Audited 464/1260 prompts
Audited 480/1260 prompts
Audited 496/1260 prompts
Audited 512/1260 prompts
Audited 528/1260 prompts
Audited 544/1260 prompts
Audited 560/1260 prompts
Audited 576/1260 prompts
Audited 592/1260 prompts
Audited 608/1260 prompts
Audited 624/1260 prompts
Audited 640/1260 prompts
Audite

CompletedProcess(args=['/usr/bin/python3', '-u', '-m', 'src.m0_knowledge_audit', '--config', 'configs/evaluation/m0_knowledge_audit_v2.json', '--output-dir', '/kaggle/working/m0-knowledge-audit-v2-n1fck5ni'], returncode=0)

In [5]:
summary = json.loads((output_dir / 'summary.json').read_text())
assert summary['execution_complete']
assert summary['predictions'] == 1260
assert summary['model_executed']
short_summary = {key: summary[key] for key in ('execution_complete', 'predictions', 'protocol_version', 'runtime', 'seconds', 'predictions_sha256')}
short_summary['answer_scoring'] = {key: value for key, value in summary['answer_scoring'].items() if key != 'by_partition_and_task'}
print(json.dumps(short_summary, indent=2))
print('Preserve summary.json, predictions.jsonl, matches_for_review.json, plan.json, environment.freeze.txt')
print('Saved output directory:', output_dir)

print('Review any token_limit_hits or format_violations before comparing models.')


{
  "execution_complete": true,
  "predictions": 1260,
  "protocol_version": "answer-only-v2",
  "runtime": {
    "cuda_runtime": "13.0",
    "device": "cuda:0",
    "dtype": "torch.float16",
    "gpu": "Tesla T4",
    "huggingface_hub": "1.33.0",
    "peak_gpu_memory_gib": 1.325,
    "torch": "2.11.0+cu130",
    "transformers": "5.20.2"
  },
  "seconds": 38.4952,
  "predictions_sha256": "3bd2761acf1e45a4396fac5b9e5a0a86d01d01df1b1c5301d0d4ca7e08180f92",
  "answer_scoring": {
    "answer_matches": 0,
    "format_violations": 0,
    "non_year_answer_matches": 0,
    "token_limit_hits": 0,
    "version": "answer-only-v2",
    "year_answer_matches": 0
  }
}
Preserve summary.json, predictions.jsonl, matches_for_review.json, plan.json, environment.freeze.txt
Saved output directory: /kaggle/working/m0-knowledge-audit-v2-n1fck5ni
Review any token_limit_hits or format_violations before comparing models.


In [6]:
import shutil

archive = shutil.make_archive(str(output_dir) + '-results', 'zip', root_dir=str(output_dir))
print('Download from the Kaggle Output panel:', archive)


Download from the Kaggle Output panel: /kaggle/working/m0-knowledge-audit-v2-n1fck5ni-results.zip


Preserve all five output files and the printed ZIP. Review raw answers, extracted answers, token-limit hits and format violations. Keep the final-test prompts and original dataset unchanged. This notebook performs inference only; v2 results remain pending until execution.